# 🚀 Server — AI Audiobook Reader (RVC) — Local

Loads the trained RVC model and starts an inference server on **localhost:8000**.

- **GPU**: NVIDIA RTX 3090 (CUDA 12)
- **Duration**: ~1–2 min to warm up
- **Run each session** while you want to test inference

Select the `vclone_venv` kernel before running.
Run Training.ipynb first — this notebook loads `models/user_voice.pth`.


In [ ]:
# ── Cell 1: Install server deps into vclone_venv ───────────────────────────────
import sys, os, subprocess, shutil

BASE_DIR   = '/home/zero/Desktop/Explore/VoiceCloning/rvc-training-inference'
APPLIO_DIR = f'{BASE_DIR}/Applio'
VENV_PY    = '/home/zero/Desktop/Explore/VoiceCloning/vclone_venv/bin/python'

uv_bin = shutil.which('uv') or '/home/zero/.local/bin/uv'

print('Installing server deps ...')
r = subprocess.run([
    uv_bin, 'pip', 'install', '--python', VENV_PY, '-q',
    'edge-tts', 'fastapi', 'uvicorn[standard]', 'pydub', 'nest_asyncio',
    '-r', f'{APPLIO_DIR}/requirements.txt',
    '--extra-index-url', 'https://download.pytorch.org/whl/cu124',
    '--index-strategy', 'unsafe-best-match',
], capture_output=True, text=True)
if r.returncode != 0:
    print(r.stderr[-2000:])
    raise RuntimeError('install failed')
print('✓ Server dependencies installed')
print(f'Python (venv): {VENV_PY}')


In [ ]:
# ── Cell 2: Config ─────────────────────────────────────────────────────────────
import sys, os, torch

BASE_DIR    = '/home/zero/Desktop/Explore/VoiceCloning/rvc-training-inference'
APPLIO_DIR  = f'{BASE_DIR}/Applio'
MODEL_PTH   = f'{BASE_DIR}/models/user_voice.pth'
MODEL_INDEX = f'{BASE_DIR}/models/user_voice.index'

assert os.path.exists(MODEL_PTH),   f'user_voice.pth not found — run Training.ipynb first'
assert os.path.exists(MODEL_INDEX), f'user_voice.index not found — run Training.ipynb first'

if torch.cuda.is_available():
    print(f'✓ GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠  No GPU — inference will be slow')

print(f'✓ Model: {MODEL_PTH}  ({os.path.getsize(MODEL_PTH)//1024//1024} MB)')
print(f'✓ Index: {MODEL_INDEX}')


In [ ]:
# ── Cell 3: Start inference server ─────────────────────────────────────────────
# Runs the FastAPI server in a subprocess (clean Python env = correct numpy/scipy).
# Blocks until Ctrl-C / Kernel → Interrupt.
import sys, os, json, re, subprocess, threading

BASE_DIR    = '/home/zero/Desktop/Explore/VoiceCloning/rvc-training-inference'
APPLIO_DIR  = f'{BASE_DIR}/Applio'
VENV_PY     = sys.executable
MODEL_PTH   = f'{BASE_DIR}/models/user_voice.pth'
MODEL_INDEX = f'{BASE_DIR}/models/user_voice.index'
PORT        = 8000

# Write config for the server subprocess
cfg_path = '/tmp/rvc_local_server_config.json'
with open(cfg_path, 'w') as f:
    json.dump({'model_pth': MODEL_PTH, 'model_index': MODEL_INDEX,
               'applio_dir': APPLIO_DIR}, f)

server_script = """
import sys, os, io, tempfile, asyncio, json

with open('/tmp/rvc_local_server_config.json') as f:
    cfg = json.load(f)
sys.path.insert(0, cfg['applio_dir'])
MODEL_PTH   = cfg['model_pth']
MODEL_INDEX = cfg['model_index']

print('[server] Importing RVC ...', flush=True)
from rvc.infer.infer import VoiceConverter
from pydub import AudioSegment
import edge_tts
print('[server] Imports done', flush=True)

vc = VoiceConverter()
print('[server] VoiceConverter ready', flush=True)

# Warmup inference (loads CUDA kernels)
async def _warmup_tts():
    comm = edge_tts.Communicate('Hello.', 'en-US-JennyNeural')
    buf = io.BytesIO()
    async for chunk in comm.stream():
        if chunk['type'] == 'audio':
            buf.write(chunk['data'])
    buf.seek(0)
    return buf.read()

mp3 = asyncio.run(_warmup_tts())
audio = AudioSegment.from_file(io.BytesIO(mp3), format='mp3')
audio = audio.set_frame_rate(40000).set_channels(1)
with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as f_in:
    audio.export(f_in.name, format='wav')
    win = f_in.name
wout = win.replace('.wav', '_rvc.wav')
vc.convert_audio(audio_input_path=win, audio_output_path=wout,
                 model_path=MODEL_PTH, index_path=MODEL_INDEX,
                 pitch=0, f0_method='rmvpe', index_rate=0.75,
                 volume_envelope=1.0, protect=0.5, export_format='WAV')
for p in (win, wout):
    if os.path.exists(p): os.unlink(p)
print('[server] Warmup complete', flush=True)

import uvicorn
from fastapi import FastAPI, Request
from fastapi.responses import Response
from fastapi.middleware.cors import CORSMiddleware
from concurrent.futures import ThreadPoolExecutor
from contextlib import asynccontextmanager

executor = ThreadPoolExecutor(max_workers=1)
EDGE_VOICE = 'en-US-JennyNeural'

async def text_to_mp3(text):
    comm = edge_tts.Communicate(text, EDGE_VOICE)
    buf = io.BytesIO()
    async for chunk in comm.stream():
        if chunk['type'] == 'audio': buf.write(chunk['data'])
    buf.seek(0)
    return buf.read()

def mp3_to_wav(mp3_bytes):
    audio = AudioSegment.from_file(io.BytesIO(mp3_bytes), format='mp3')
    audio = audio.set_frame_rate(40000).set_channels(1)
    buf = io.BytesIO(); audio.export(buf, format='wav'); buf.seek(0)
    return buf.read()

def rvc_convert(wav_bytes):
    with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as f:
        f.write(wav_bytes); in_path = f.name
    out_path = in_path.replace('.wav', '_rvc.wav')
    try:
        vc.convert_audio(audio_input_path=in_path, audio_output_path=out_path,
                         model_path=MODEL_PTH, index_path=MODEL_INDEX,
                         pitch=0, f0_method='rmvpe', index_rate=0.75,
                         volume_envelope=1.0, protect=0.5, export_format='WAV')
        with open(out_path, 'rb') as f: return f.read()
    finally:
        for p in (in_path, out_path):
            if os.path.exists(p): os.unlink(p)

@asynccontextmanager
async def lifespan(app):
    print('SERVER_READY', flush=True)
    yield

app = FastAPI(lifespan=lifespan)
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_methods=['*'], allow_headers=['*'])

@app.get('/ping')
def ping(): return {'status': 'ready', 'model': 'rvc_local'}

@app.post('/synth')
async def synth(request: Request):
    body = await request.json()
    text = body.get('text', '').strip()
    if not text: return Response(b'', media_type='audio/wav')
    loop = asyncio.get_running_loop()
    mp3_bytes = await text_to_mp3(text)
    wav_bytes = await loop.run_in_executor(executor, mp3_to_wav, mp3_bytes)
    rvc_bytes = await loop.run_in_executor(executor, rvc_convert, wav_bytes)
    return Response(rvc_bytes, media_type='audio/wav')

uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning')
"""

with open('/tmp/rvc_local_server.py', 'w') as f:
    f.write(server_script.lstrip())

print(f'Starting server on http://localhost:{PORT} ...')
print('(~1-2 min for model load + warmup)\n')

proc = subprocess.Popen(
    [VENV_PY, '/tmp/rvc_local_server.py'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, cwd=APPLIO_DIR
)

ready = False
for line in proc.stdout:
    print(line, end='', flush=True)
    if 'SERVER_READY' in line:
        ready = True
        break

if not ready:
    proc.wait()
    raise RuntimeError(f'Server exited before ready (code {proc.returncode})')

def _fwd():
    for line in proc.stdout:
        print(line, end='', flush=True)
threading.Thread(target=_fwd, daemon=True).start()

print(f'\n✅ Server ready at http://localhost:{PORT}')
print(f'   /ping  → GET  http://localhost:{PORT}/ping')
print(f'   /synth → POST http://localhost:{PORT}/synth  body: ' + '{"text": "..."}')
print('\nRun Cell 4 to test, or use curl:')
print(f'  curl http://localhost:{PORT}/ping')
print('  curl -X POST http://localhost:' + str(PORT) + '/synth -H "Content-Type: application/json" -d \'{"text":"Hello world"}\' -o test.wav')
print('\nKeep this cell running. Interrupt kernel to stop the server.')

proc.wait()   # blocks until Ctrl-C / Kernel Interrupt


In [ ]:
# ── Cell 4: Smoke tests — run while Cell 3 is blocked in another tab/session ───
import requests

BASE = 'http://localhost:8000'

r = requests.get(f'{BASE}/ping', timeout=10)
assert r.ok and r.json().get('status') == 'ready', f'Ping failed: {r.text}'
print(f'✓ /ping  OK: {r.json()}')

r2 = requests.post(f'{BASE}/synth', json={'text': 'Hello, this is a voice clone test.'}, timeout=60)
assert r2.ok, f'/synth failed: {r2.status_code} {r2.text}'
assert len(r2.content) > 1000, f'Audio too short: {len(r2.content)} bytes'
assert r2.content[:4] == b'RIFF', 'Response is not a valid WAV'
print(f'✓ /synth OK: {len(r2.content):,} bytes of audio')

# Save to file for listening
out_path = '/home/zero/Desktop/Explore/VoiceCloning/rvc-training-inference/test_output.wav'
with open(out_path, 'wb') as f:
    f.write(r2.content)
print(f'✓ Saved to {out_path} — play with: aplay {out_path}')
